In [6]:
pip install streamlit

Defaulting to user installation because normal site-packages is not writeable
  Using cached streamlit-1.55.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached cachetools-7.0.5-py3-none-any.whl.metadata (5.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached narwhals-2.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
Using cached streamlit-1.55.0-py3-none-any.whl (9.1 MB)
Using cached altair-6.0.0-py3-none-any.whl (795 kB)
Using cached cachetools-7.0.5-py3-none-any.whl (13 kB)
Using cached gitpython-3.1.46-py3

  You can safely remove it manually.


In [ ]:
import gradio as gr
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

# ---------------------------
# LLM INIT (Mistral)
# ---------------------------
model = init_chat_model(model="mistral-medium")

# ---------------------------
# RBAC CONFIG
# ---------------------------
PERMISSIONS = {
    "student": [
        "view_own_attendance",
        "view_exam_results"
    ],
    "faculty": [
        "view_own_attendance",
        "view_others_attendance",
        "update_exam_results",
        "view_all_records"
    ],
    "admin": [
        "view_own_attendance",
        "view_others_attendance",
        "update_exam_results",
        "approve_course_changes",
        "view_all_records"
    ]
}

# ---------------------------
# ACTION DETECTION
# ---------------------------
def detect_action(user_input: str):
    text = user_input.lower()

    if "attendance" in text and "my" in text:
        return "view_own_attendance"
    elif "attendance" in text:
        return "view_others_attendance"
    elif "exam" in text or "result" in text:
        return "update_exam_results"
    elif "approve" in text or "registration" in text:
        return "approve_course_changes"
    elif "all student" in text or "records" in text:
        return "view_all_records"
    return "unknown"


# ---------------------------
# MOCK DATA
# ---------------------------
DATA = {
    "attendance": "Your attendance is 82%",
    "results": "Your latest result: AI - 88 marks",
    "all_records": "Displaying all student records..."
}

# ---------------------------
# CORE HANDLER
# ---------------------------
def handle_query(role, message, history):
    if message.lower() in ["hi", "hello", "hii"]:
        reply = f"You are logged in as {role.upper()}.\n\nYou can access:\n- " + "\n- ".join(PERMISSIONS[role])
        history.append((message, reply))
        return history, ""

    action = detect_action(message)

    if action == "unknown":
        response = model.invoke([HumanMessage(content=message)])
        reply = response.content
        history.append((message, reply))
        return history, ""

    if action not in PERMISSIONS[role]:
        reply = f"❌ Access Denied: '{action}' not allowed for {role}"
        history.append((message, reply))
        return history, ""

    # Authorized actions
    if action == "view_own_attendance":
        reply = DATA["attendance"]

    elif action == "view_others_attendance":
        reply = "Showing attendance of all students..."

    elif action == "update_exam_results":
        reply = "Exam results updated successfully."

    elif action == "approve_course_changes":
        reply = "Course registration approved."

    elif action == "view_all_records":
        reply = DATA["all_records"]

    else:
        reply = "Action executed."

    history.append((message, reply))
    return history, ""


# ---------------------------
# GRADIO UI
# ---------------------------
with gr.Blocks(title="University Smart Assistant") as demo:
    gr.Markdown("## 🎓 University Smart Assistant (RBAC + Mistral)")

    role = gr.Dropdown(
        choices=["student", "faculty", "admin"],
        label="Select Role",
        value="student"
    )

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask something...")
    clear = gr.Button("Clear Chat")

    msg.submit(handle_query, inputs=[role, msg, chatbot], outputs=[chatbot, msg])
    clear.click(lambda: [], None, chatbot)

# Launch
demo.launch()